# 🔣 Exercícios — Lógica Proposicional e de Primeira Ordem

**Disciplina:** Inteligência Artificial | **Nível:** Intermediário

> Pratique tabelas-verdade, avaliação de fórmulas e inferência lógica com Python.


## 1. Tabelas-Verdade

Vamos construir tabelas-verdade programaticamente para verificar fórmulas lógicas.

In [ ]:
from itertools import product

def tabela_verdade(variaveis, formula_fn, nome_formula=""):
    """Gera e imprime a tabela-verdade de uma fórmula lógica."""
    n = len(variaveis)
    header = "  ".join(f"{v:^5}" for v in variaveis) + "  |  " + f"{nome_formula:^15}"
    print(header)
    print("-" * len(header))
    resultados = []
    for valores in product([False, True], repeat=n):
        env = dict(zip(variaveis, valores))
        res = formula_fn(**env)
        linha = "  ".join(f"{'V' if v else 'F':^5}" for v in valores)
        print(f"{linha}  |  {'V' if res else 'F':^15}")
        resultados.append(res)
    print()
    return resultados

# Exemplo: P ∧ Q (conjunção)
tabela_verdade(['P','Q'], lambda P,Q: P and Q, "P ∧ Q")

# Exemplo: P → Q (implicação)  
tabela_verdade(['P','Q'], lambda P,Q: (not P) or Q, "P → Q")


### 📝 Exercício 1

Gere a tabela-verdade para as seguintes fórmulas:
1. `P ∨ (¬Q)` (disjunção com negação)
2. `(P → Q) ∧ (Q → P)` (bicondicional via implicações)
3. `(P ∧ Q) → (P ∨ Q)` — é uma **tautologia**?

In [ ]:
# ✏️ Complete aqui:

# 1. P ∨ (¬Q)
tabela_verdade(['P','Q'], lambda P,Q: ..., "P ∨ ¬Q")

# 2. (P → Q) ∧ (Q → P)
tabela_verdade(['P','Q'], lambda P,Q: ..., "(P→Q)∧(Q→P)")

# 3. (P ∧ Q) → (P ∨ Q)
res = tabela_verdade(['P','Q'], lambda P,Q: ..., "(P∧Q)→(P∨Q)")
print("Tautologia?", all(res))


## 2. Verificação de Satisfatibilidade

Uma fórmula é **satisfatível** se existe ao menos uma atribuição que a torna verdadeira.

In [ ]:
def e_satisfativel(variaveis, formula_fn):
    for valores in product([False, True], repeat=len(variaveis)):
        env = dict(zip(variaveis, valores))
        if formula_fn(**env):
            return True, dict(zip(variaveis, valores))
    return False, {}

def e_tautologia(variaveis, formula_fn):
    return all(formula_fn(**dict(zip(variaveis, v)))
               for v in product([False, True], repeat=len(variaveis)))

# Testes
formulas = [
    (['P','Q'], lambda P,Q: (not P) or Q,              "P → Q"),
    (['P'],     lambda P:   P and (not P),              "P ∧ ¬P"),
    (['P'],     lambda P:   P or  (not P),              "P ∨ ¬P"),
    (['P','Q'], lambda P,Q: ((not P) or Q) and P and (not Q), "(P→Q)∧P∧¬Q"),
]

print(f"{'Fórmula':<25} | {'Satisfatível':^12} | {'Tautologia':^10}")
print("-" * 55)
for vars_, fn, nome in formulas:
    sat, wit = e_satisfativel(vars_, fn)
    tau = e_tautologia(vars_, fn)
    print(f"{nome:<25} | {'Sim'+str(wit) if sat else 'Não':^12} | {'Sim' if tau else 'Não':^10}")


## 3. Lógica de Primeira Ordem — Prolog-like em Python

Vamos simular um mini motor de inferência com predicados e regras.

In [ ]:
# Base de fatos: humano(X) e mortal(X)
fatos = {
    "humano": {"Sócrates", "Platão", "Aristóteles"},
    "filosofo": {"Sócrates", "Platão", "Aristóteles", "Nietzsche"},
    "mortal": set()
}

# Regras de inferência
def aplicar_regras():
    """mortal(X) :- humano(X)"""
    novos = set()
    for x in fatos["humano"]:
        if x not in fatos["mortal"]:
            novos.add(x)
    fatos["mortal"].update(novos)
    return novos

print("Antes das regras:", fatos)
novos = aplicar_regras()
print("Novos fatos derivados:", novos)
print("Após inferência:", fatos)

# Consulta
def consulta(predicado, individuo):
    return individuo in fatos.get(predicado, set())

print("\nConsultas:")
print(f"  mortal(Sócrates)? {consulta('mortal','Sócrates')}")
print(f"  mortal(Nietzsche)? {consulta('mortal','Nietzsche')}")


### 📝 Exercício 3

Adicione as seguintes regras à base de conhecimento acima:
1. `sábio(X) :- filosofo(X) ∧ humano(X)` (filósofo humano é sábio)
2. `legado(X) :- sábio(X)` (sábio deixa legado)

Depois, consulte: *Nietzsche deixou legado?* (ele é filósofo mas não humano na base — interessante!)

In [ ]:
# ✏️ Adicione as regras e as consultas aqui:
fatos["sabio"]  = set()
fatos["legado"] = set()

def regra_sabio():
    # TODO: X é sábio se é filósofo E humano
    pass

def regra_legado():
    # TODO: X tem legado se é sábio
    pass

regra_sabio()
regra_legado()
print("Sábios:", fatos["sabio"])
print("Legados:", fatos["legado"])
print("Nietzsche tem legado?", consulta('legado', 'Nietzsche'))


## 4. Resolução e Prova por Refutação

A **resolução** é o método fundamental de prova em lógica de primeira ordem.

In [ ]:
# Exemplo de prova por refutação (resolução)
# Provar: Sócrates é mortal
# Fatos: humano(Sócrates), ∀X: humano(X) → mortal(X)
# Negação do objetivo: ¬mortal(Sócrates)
# Esperamos: contradição!

clausulas = [
    frozenset(["humano_Socrates"]),            # humano(Sócrates)
    frozenset(["mortal_X", "¬humano_X"]),      # ∀X: ¬humano(X) ∨ mortal(X)  i.e. humano→mortal
    frozenset(["¬mortal_Socrates"]),            # negação do objetivo
]

print("Cláusulas iniciais:")
for c in clausulas: print(" ", c)

# Resolução manual (simplificada para este exemplo)
# Resolver cláusula 1 e 2 (instanciar X=Sócrates):
resolvente1 = frozenset(["mortal_Socrates"])
print(f"\nResolvendo {clausulas[0]} com {clausulas[1]}:")
print(f"  → {resolvente1}")

# Resolver resolvente1 com cláusula 3:
resolvente2 = frozenset()  # conjunto vazio = contradição!
print(f"\nResolvendo {resolvente1} com {clausulas[2]}:")
print(f"  → {resolvente2}")

print("\n✅ Contradição encontrada! Portanto: Sócrates É mortal. ∎")


### 📝 Exercício Final

Usando a mesma estrutura, prove por refutação:

- *Platão é filósofo humano*
- *Todo filósofo humano é sábio*
- **Portanto: Platão é sábio**

Escreva as cláusulas e o processo de resolução.

In [ ]:
# ✏️ Sua prova por refutação aqui:
clausulas_ex = [
    # TODO: adicione as cláusulas
]
# TODO: mostre o processo de resolução
